# 12_cross_window_protocol

This notebook wraps the strict cross-window experiment.


In [ ]:
from pathlib import Path
import sys, pandas as pd
ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path: sys.path.append(str(ROOT))
from src.pipeline.cross_window_experiment import CrossWindowSplitConfig, run_cross_window_experiment
from src.pipeline.formal_experiment import RuleMiningConfig, WarmStartConfig, EvalConfig
plan_df = pd.read_csv(ROOT / "data/stream/swat/formal_mixed_windows_plan.csv")
summary_df, splits, train_acc = run_cross_window_experiment(
    project_root=ROOT,
    plan_df=plan_df,
    split_config=CrossWindowSplitConfig(train_per_bin=4, val_per_bin=1, test_per_bin=1, random_state=42),
    mining_config=RuleMiningConfig(),
    warmstart_config=WarmStartConfig(),
    eval_config=EvalConfig(),
)
print('splits =', splits)
print('global_train_accuracy =', train_acc)
display(summary_df)


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.pipeline.cross_window_experiment import CrossWindowSplitConfig, run_cross_window_experiment
from src.pipeline.formal_experiment import RuleMiningConfig, WarmStartConfig, EvalConfig

plan_df = pd.read_csv(ROOT / "data/stream/swat/formal_mixed_windows_plan.csv")

rows = []

for seed in [0, 1, 2, 3, 4]:
    summary_df, splits, train_acc = run_cross_window_experiment(
        project_root=ROOT,
        plan_df=plan_df,
        split_config=CrossWindowSplitConfig(
            train_per_bin=4,
            val_per_bin=1,
            test_per_bin=1,
            random_state=seed
        ),
        mining_config=RuleMiningConfig(),
        warmstart_config=WarmStartConfig(),
        eval_config=EvalConfig(),
    )

    rows.append({
        "seed": seed,
        "num_test_windows": len(summary_df),
        "global_train_accuracy": train_acc,
        "mean_attack_keep_rate": summary_df["attack_keep_rate"].mean(),
        "mean_normal_disable_rate": summary_df["normal_disable_rate"].mean(),
        "mean_selection_accuracy": summary_df["selection_accuracy"].mean(),
        "mean_greedy_total_reward": summary_df["greedy_total_reward"].mean(),
    })

repeat_df = pd.DataFrame(rows)
print(repeat_df)

print("\n[mean over seeds]")
print(repeat_df.mean(numeric_only=True))

print("\n[std over seeds]")
print(repeat_df.std(numeric_only=True))

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.pipeline.cross_window_experiment import CrossWindowSplitConfig, run_cross_window_experiment
from src.pipeline.formal_experiment import RuleMiningConfig, WarmStartConfig, EvalConfig

plan_df = pd.read_csv(ROOT / "data/stream/swat/formal_mixed_windows_plan.csv")

rows = []

for seed in [0, 1, 2, 3, 4]:
    summary_df, splits, train_acc = run_cross_window_experiment(
        project_root=ROOT,
        plan_df=plan_df,
        split_config=CrossWindowSplitConfig(
            train_per_bin=4,
            val_per_bin=1,
            test_per_bin=1,
            random_state=seed
        ),
        mining_config=RuleMiningConfig(),
        warmstart_config=WarmStartConfig(),
        eval_config=EvalConfig(),
    )

    rows.append({
        "seed": seed,
        "num_test_windows": len(summary_df),
        "global_train_accuracy": train_acc,
        "mean_attack_keep_rate": summary_df["attack_keep_rate"].mean(),
        "mean_normal_disable_rate": summary_df["normal_disable_rate"].mean(),
        "mean_selection_accuracy": summary_df["selection_accuracy"].mean(),
        "mean_greedy_total_reward": summary_df["greedy_total_reward"].mean(),
    })

repeat_df = pd.DataFrame(rows)

save_path = ROOT / "outputs/logs/cross_window_repeat5_summary.csv"
repeat_df.to_csv(save_path, index=False)

print("saved:", save_path)
print(repeat_df)

print("\n[mean over seeds]")
print(repeat_df.mean(numeric_only=True))

print("\n[std over seeds]")
print(repeat_df.std(numeric_only=True))

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.pipeline.cross_window_experiment import CrossWindowSplitConfig, run_cross_window_experiment
from src.pipeline.formal_experiment import RuleMiningConfig, WarmStartConfig, EvalConfig

plan_df = pd.read_csv(ROOT / "data/stream/swat/formal_mixed_windows_plan.csv")

rows = []
all_test_ids = []

for seed in [0, 1, 2, 3, 4]:
    summary_df, splits, train_acc = run_cross_window_experiment(
        project_root=ROOT,
        plan_df=plan_df,
        split_config=CrossWindowSplitConfig(
            train_per_bin=3,
            val_per_bin=1,
            test_per_bin=2,
            random_state=seed
        ),
        mining_config=RuleMiningConfig(),
        warmstart_config=WarmStartConfig(),
        eval_config=EvalConfig(),
    )

    test_ids = splits["test"]
    all_test_ids.extend(test_ids)

    rows.append({
        "seed": seed,
        "test_ids": test_ids,
        "num_test_windows": len(test_ids),
        "mean_selection_accuracy": summary_df["selection_accuracy"].mean(),
        "mean_normal_disable_rate": summary_df["normal_disable_rate"].mean(),
    })

coverage_df = pd.DataFrame(rows)
unique_test_ids = sorted(set(all_test_ids))

print("[coverage_df]")
print(coverage_df)

print("\nunique_test_ids =", unique_test_ids)
print("num_unique_test_windows =", len(unique_test_ids))

cover_plan = plan_df[plan_df["window_id"].isin(unique_test_ids)].copy()
if "attack_ratio" in cover_plan.columns:
    cover_plan = cover_plan.sort_values("attack_ratio").reset_index(drop=True)

print("\n[covered test windows in plan]")
print(cover_plan[["window_id", "attack_ratio", "attack_bin"]])

In [ ]:
from pathlib import Path
import sys, pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.pipeline.cross_window_experiment import CrossWindowSplitConfig, run_cross_window_experiment
from src.pipeline.formal_experiment import RuleMiningConfig, WarmStartConfig, EvalConfig

plan_df = pd.read_csv(ROOT / "data/stream/swat/formal_mixed_windows_plan.csv")

summary_df, splits, train_acc = run_cross_window_experiment(
    project_root=ROOT,
    plan_df=plan_df,
    split_config=CrossWindowSplitConfig(
        train_per_bin=3,
        val_per_bin=1,
        test_per_bin=2,
        random_state=42
    ),
    mining_config=RuleMiningConfig(),
    warmstart_config=WarmStartConfig(),
    eval_config=EvalConfig(),
)

print("splits =", splits)
print("global_train_accuracy =", train_acc)
display(summary_df)

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.pipeline.cross_window_experiment import CrossWindowSplitConfig, run_cross_window_experiment
from src.pipeline.formal_experiment import RuleMiningConfig, WarmStartConfig, EvalConfig

plan_df = pd.read_csv(ROOT / "data/stream/swat/formal_mixed_windows_plan.csv")

rows = []
all_test_ids = []

for seed in [0, 1, 2, 3, 4]:
    summary_df, splits, train_acc = run_cross_window_experiment(
        project_root=ROOT,
        plan_df=plan_df,
        split_config=CrossWindowSplitConfig(
            train_per_bin=3,
            val_per_bin=1,
            test_per_bin=2,
            random_state=seed
        ),
        mining_config=RuleMiningConfig(),
        warmstart_config=WarmStartConfig(),
        eval_config=EvalConfig(),
    )

    test_ids = splits["test"]
    all_test_ids.extend(test_ids)

    rows.append({
        "seed": seed,
        "num_train_windows": len(splits["train"]),
        "num_val_windows": len(splits["val"]),
        "num_test_windows": len(splits["test"]),
        "test_ids": test_ids,
        "global_train_accuracy": train_acc,
        "mean_attack_keep_rate": summary_df["attack_keep_rate"].mean(),
        "mean_normal_disable_rate": summary_df["normal_disable_rate"].mean(),
        "mean_selection_accuracy": summary_df["selection_accuracy"].mean(),
        "mean_greedy_total_reward": summary_df["greedy_total_reward"].mean(),
    })

repeat_df = pd.DataFrame(rows)
save_path = ROOT / "outputs/logs/cross_window_repeat5_strict_summary.csv"
repeat_df.to_csv(save_path, index=False)

unique_test_ids = sorted(set(all_test_ids))
cover_plan = plan_df[plan_df["window_id"].isin(unique_test_ids)].copy()
if "attack_ratio" in cover_plan.columns:
    cover_plan = cover_plan.sort_values("attack_ratio").reset_index(drop=True)

print("saved:", save_path)
print("\n[repeat_df]")
print(repeat_df)

print("\n[mean over seeds]")
print(repeat_df.mean(numeric_only=True))

print("\n[std over seeds]")
print(repeat_df.std(numeric_only=True))

print("\nunique_test_ids =", unique_test_ids)
print("num_unique_test_windows =", len(unique_test_ids))

print("\n[covered test windows in plan]")
print(cover_plan[["window_id", "attack_ratio", "attack_bin"]])

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.pipeline.cross_window_experiment import CrossWindowSplitConfig, run_cross_window_experiment
from src.pipeline.formal_experiment import RuleMiningConfig, WarmStartConfig, EvalConfig

plan_df = pd.read_csv(ROOT / "data/stream/swat/formal_mixed_windows_plan.csv")

# 单独分析最差的 seed=4
summary_df, splits, train_acc = run_cross_window_experiment(
    project_root=ROOT,
    plan_df=plan_df,
    split_config=CrossWindowSplitConfig(
        train_per_bin=3,
        val_per_bin=1,
        test_per_bin=2,
        random_state=4
    ),
    mining_config=RuleMiningConfig(),
    warmstart_config=WarmStartConfig(),
    eval_config=EvalConfig(),
)

print("splits =", splits)
print("global_train_accuracy =", train_acc)
print(summary_df.sort_values("attack_ratio").reset_index(drop=True))

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.pipeline.cross_window_experiment import CrossWindowSplitConfig, run_cross_window_experiment
from src.pipeline.formal_experiment import RuleMiningConfig, WarmStartConfig, EvalConfig

plan_df = pd.read_csv(ROOT / "data/stream/swat/formal_mixed_windows_plan.csv")

rows = []
all_test_ids = []

for seed in [0, 1, 2, 3, 4]:
    summary_df, splits, train_acc = run_cross_window_experiment(
        project_root=ROOT,
        plan_df=plan_df,
        split_config=CrossWindowSplitConfig(
            train_per_bin=3,
            val_per_bin=1,
            test_per_bin=2,
            random_state=seed
        ),
        mining_config=RuleMiningConfig(),
        warmstart_config=WarmStartConfig(),
        eval_config=EvalConfig(),
    )

    test_ids = splits["test"]
    all_test_ids.extend(test_ids)

    rows.append({
        "seed": seed,
        "num_train_windows": len(splits["train"]),
        "num_val_windows": len(splits["val"]),
        "num_test_windows": len(splits["test"]),
        "test_ids": test_ids,
        "global_train_accuracy": train_acc,
        "mean_attack_keep_rate": summary_df["attack_keep_rate"].mean(),
        "mean_normal_disable_rate": summary_df["normal_disable_rate"].mean(),
        "mean_selection_accuracy": summary_df["selection_accuracy"].mean(),
        "mean_greedy_total_reward": summary_df["greedy_total_reward"].mean(),
    })

repeat_df = pd.DataFrame(rows)
save_path = ROOT / "outputs/logs/cross_window_repeat5_strict_summary.csv"
repeat_df.to_csv(save_path, index=False)

unique_test_ids = sorted(set(all_test_ids))
cover_plan = plan_df[plan_df["window_id"].isin(unique_test_ids)].copy()
if "attack_ratio" in cover_plan.columns:
    cover_plan = cover_plan.sort_values("attack_ratio").reset_index(drop=True)

print("saved:", save_path)
print("\n[repeat_df]")
print(repeat_df)

print("\n[mean over seeds]")
print(repeat_df.mean(numeric_only=True))

print("\n[std over seeds]")
print(repeat_df.std(numeric_only=True))

print("\nunique_test_ids =", unique_test_ids)
print("num_unique_test_windows =", len(unique_test_ids))

print("\n[covered test windows in plan]")
print(cover_plan[["window_id", "attack_ratio", "attack_bin"]])

In [ ]:
from pathlib import Path
import sys, pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.pipeline.cross_window_experiment import CrossWindowSplitConfig, run_cross_window_experiment
from src.pipeline.formal_experiment import RuleMiningConfig, WarmStartConfig, EvalConfig

plan_df = pd.read_csv(ROOT / "data/stream/swat/formal_mixed_windows_plan.csv")

summary_df, splits, train_acc = run_cross_window_experiment(
    project_root=ROOT,
    plan_df=plan_df,
    split_config=CrossWindowSplitConfig(
        train_per_bin=3,
        val_per_bin=1,
        test_per_bin=2,
        random_state=42
    ),
    mining_config=RuleMiningConfig(),
    warmstart_config=WarmStartConfig(),
    eval_config=EvalConfig(),
)

print("splits =", splits)
print("global_train_accuracy =", train_acc)
display(summary_df)

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.pipeline.cross_window_experiment import CrossWindowSplitConfig, run_cross_window_experiment
from src.pipeline.formal_experiment import RuleMiningConfig, WarmStartConfig, EvalConfig

plan_df = pd.read_csv(ROOT / "data/stream/swat/formal_mixed_windows_plan.csv")

all_rows = []

for seed in [0, 1, 2, 3, 4]:
    summary_df, splits, train_acc = run_cross_window_experiment(
        project_root=ROOT,
        plan_df=plan_df,
        split_config=CrossWindowSplitConfig(
            train_per_bin=3,
            val_per_bin=1,
            test_per_bin=2,
            random_state=seed
        ),
        mining_config=RuleMiningConfig(),
        warmstart_config=WarmStartConfig(),
        eval_config=EvalConfig(),
    )

    summary_df = summary_df.copy()
    summary_df["seed"] = seed
    all_rows.append(summary_df)

all_df = pd.concat(all_rows, axis=0, ignore_index=True)

window_stats = all_df.groupby("window_id")[[
    "attack_ratio",
    "attack_keep_rate",
    "normal_disable_rate",
    "selection_accuracy",
    "greedy_total_reward"
]].agg(["mean", "std", "count"])

window_stats_path = ROOT / "outputs/logs/cross_window_window_stats.csv"
window_stats.to_csv(window_stats_path)
 
print("saved:", window_stats_path)
print(window_stats.sort_values(("selection_accuracy", "mean")))